# TMDB Recommendation Engine Experiments (Python)

Goal: iterate quickly on a fast, high-quality blending strategy ("People also search for" style) that we can productionize in the Python microservice and wire into the app.

## Setup
- Create a TMDB API key (v4 bearer token recommended, v3 key also works).
- In VS Code, set an environment variable before running:
  - Windows PowerShell: `$env:TMDB_API_KEY="<your_token_or_key>"`
  - bash: `export TMDB_API_KEY="<your_token_or_key>"`
- This notebook prefers Bearer (v4) if the token starts with `ey`, otherwise falls back to `api_key` param.

In [7]:
# Imports and environment
import os, math, time, json
from typing import Dict, List, Any, Tuple

try:
    import pandas as pd
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas'])
    import pandas as pd

import requests

TMDB_API_KEY = os.getenv('TMDB_API_KEY') or os.getenv('TMDB_API_KEY_V3') or ''
assert TMDB_API_KEY, 'Please set TMDB_API_KEY or TMDB_API_KEY_V3 in your environment.'

BASE_URL = 'https://api.themoviedb.org/3'
def _headers_and_params():
    headers = {}
    params = {}
    if TMDB_API_KEY.startswith('ey'):
        headers['Authorization'] = f'Bearer {TMDB_API_KEY}'
        headers['Content-Type'] = 'application/json'
    else:
        params['api_key'] = TMDB_API_KEY
    return headers, params

def tmdb_get(path: str, extra_params: Dict[str, str] | None = None) -> Any:
    headers, params = _headers_and_params()
    if extra_params:
        params.update(extra_params)
    url = f'{BASE_URL}{path}'
    r = requests.get(url, headers=headers, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

def get_movie_details(movie_id: int) -> Dict[str, Any]:
    return tmdb_get(f'/movie/{movie_id}', {'append_to_response': 'credits,keywords'})

def get_movie_similar(movie_id: int) -> List[Dict[str, Any]]:
    data = tmdb_get(f'/movie/{movie_id}/similar')
    return data.get('results', [])

def get_movie_recommendations(movie_id: int) -> List[Dict[str, Any]]:
    data = tmdb_get(f'/movie/{movie_id}/recommendations')
    return data.get('results', [])

def get_tv_details(tv_id: int) -> Dict[str, Any]:
    return tmdb_get(f'/tv/{tv_id}', {'append_to_response': 'aggregate_credits'})

def get_tv_similar(tv_id: int) -> List[Dict[str, Any]]:
    data = tmdb_get(f'/tv/{tv_id}/similar')
    return data.get('results', [])

def get_tv_recommendations(tv_id: int) -> List[Dict[str, Any]]:
    data = tmdb_get(f'/tv/{tv_id}/recommendations')
    return data.get('results', [])

In [8]:
# Feature helpers
def get_year(item: Dict[str, Any]) -> int | None:
    date_key = 'release_date' if 'release_date' in item else 'first_air_date'
    date_str = item.get(date_key)
    if not date_str:
        return None
    try:
        return int(date_str.split('-')[0])
    except Exception:
        return None

def get_genre_ids(item: Dict[str, Any]) -> List[int]:
    if 'genre_ids' in item and isinstance(item['genre_ids'], list):
        return [int(g) for g in item['genre_ids']]
    if 'genres' in item and isinstance(item['genres'], list):
        return [int(g.get('id')) for g in item['genres'] if 'id' in g]
    return []

def genre_overlap(a: List[int], b: List[int]) -> int:
    sa, sb = set(a), set(b)
    return len(sa & sb)

def clamp(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))

In [9]:
# Scoring configuration (tweak weights here)
WEIGHTS = {
    'genre_overlap': 0.52,
    'year_proximity': 0.10,
    'rating': 0.24,
    'popularity': 0.14,
}

QUALITY_FLOORS = {
    'vote_count': 300,
    'vote_average': 6.2,
}

def score_movie_candidate(source: Dict[str, Any], cand: Dict[str, Any]) -> Tuple[float, Dict[str, float]]:
    # Floors
    va = float(cand.get('vote_average') or 0.0)
    vc = float(cand.get('vote_count') or 0.0)
    if vc < QUALITY_FLOORS['vote_count'] or va < QUALITY_FLOORS['vote_average']:
        return 0.0, {'floor_filter': 1.0}

    src_genres = get_genre_ids(source)
    tgt_genres = get_genre_ids(cand)
    go = genre_overlap(src_genres, tgt_genres)  # 0..N
    go_norm = clamp(go / 4.0, 0.0, 1.0)  # normalize assuming 4+ overlap is excellent

    src_year = get_year(source)
    tgt_year = get_year(cand)
    if src_year and tgt_year:
        year_diff = abs(src_year - tgt_year)
        # within 5 years => strong, 10 years => moderate; cap at 20
        yp = 1.0 - clamp(year_diff / 20.0, 0.0, 1.0)
    else:
        yp = 0.5

    rating_term = clamp((va - 5.0) / 3.0, 0.0, 1.0)  # 5->8 mapped to 0..1
    pop = float(cand.get('popularity') or 0.0)
    pop_term = clamp(pop / 200.0, 0.0, 1.0)

    components = {
        'genre_overlap': go_norm,
        'year_proximity': yp,
        'rating': rating_term,
        'popularity': pop_term,
    }
    score = sum(WEIGHTS[k] * components[k] for k in WEIGHTS)
    return float(score), components

In [10]:
# Recommender for movies
def recommend_for_movie(movie_id: int, k: int = 20) -> pd.DataFrame:
    src = get_movie_details(movie_id)
    sim = get_movie_similar(movie_id)
    rec = get_movie_recommendations(movie_id)
    pool = {m['id']: m for m in (sim + rec) if isinstance(m, dict) and m.get('id')}

    rows = []
    for cid, cand in pool.items():
        if cid == movie_id:
            continue
        score, comps = score_movie_candidate(src, cand)
        if score <= 0:
            continue
        rows.append({
            'id': cid,
            'title': cand.get('title') or cand.get('name'),
            'score': score,
            'vote_average': cand.get('vote_average'),
            'vote_count': cand.get('vote_count'),
            'popularity': cand.get('popularity'),
            'genre_overlap': comps['genre_overlap'],
            'year_proximity': comps['year_proximity'],
        })

    df = pd.DataFrame(rows).sort_values('score', ascending=False).head(k)
    return df.reset_index(drop=True)

# Recommender for TV (uses similar pattern)
def recommend_for_tv(tv_id: int, k: int = 20) -> pd.DataFrame:
    src = get_tv_details(tv_id)
    sim = get_tv_similar(tv_id)
    rec = get_tv_recommendations(tv_id)
    pool = {m['id']: m for m in (sim + rec) if isinstance(m, dict) and m.get('id')}

    rows = []
    for cid, cand in pool.items():
        if cid == tv_id:
            continue
        score, comps = score_movie_candidate(src, cand)
        if score <= 0:
            continue
        rows.append({
            'id': cid,
            'name': cand.get('name') or cand.get('title'),
            'score': score,
            'vote_average': cand.get('vote_average'),
            'vote_count': cand.get('vote_count'),
            'popularity': cand.get('popularity'),
            'genre_overlap': comps['genre_overlap'],
            'year_proximity': comps['year_proximity'],
        })

    df = pd.DataFrame(rows).sort_values('score', ascending=False).head(k)
    return df.reset_index(drop=True)

In [11]:
# Quick smoke tests (requires valid TMDB_API_KEY)
MOVIE_SEEDS = {
    'Interstellar': 157336,
    'Inception': 27205,
    'The Dark Knight': 155,
}

for name, mid in MOVIE_SEEDS.items():
    print(f'\nTop picks for {name} ({mid})')
    disp = recommend_for_movie(mid, k=10)
    display(disp)

TV_SEEDS = {
    'Breaking Bad': 1396,
    'Game of Thrones': 1399,
}
for name, tid in TV_SEEDS.items():
    print(f'\nTop picks for {name} ({tid})')
    disp = recommend_for_tv(tid, k=10)
    display(disp)


Top picks for Interstellar (157336)


,id,title,score,vote_average,vote_count,popularity,genre_overlap,year_proximity
0,286217,The Martian,0.712292,7.700,20443,16.1315,0.75,0.95
1,27205,Inception,0.599646,8.370,37994,28.0656,0.50,0.80
2,118340,Guardians of the Galaxy,0.597202,7.905,28893,6.8599,0.50,1.00
3,76341,Mad Max: Fury Road,0.573161,7.626,23382,11.5439,0.50,0.95
4,1124,The Prestige,0.566132,8.204,16817,8.7604,0.50,0.60
5,49047,Gravity,0.533977,7.162,15813,8.5962,0.50,0.95
6,131631,The Hunger Games: Mockingjay - Part 1,0.513847,6.815,16285,12.3526,0.50,1.00
7,11,Star Wars,0.510903,8.200,21513,15.5750,0.50,0.00
8,105,Back to the Future,0.508391,8.321,20889,11.9872,0.50,0.00
9,253412,Everest,0.504058,6.825,5040,4.3684,0.50,0.95



Top picks for Inception (27205)


,id,title,score,vote_average,vote_count,popularity,genre_overlap,year_proximity
0,19995,Avatar,0.711381,7.593,32633,27.0592,0.75,0.95
1,70160,The Hunger Games,0.664238,7.218,22643,9.7116,0.75,0.90
2,68721,Iron Man 3,0.639099,6.930,22831,13.8550,0.75,0.85
3,157336,Interstellar,0.628621,8.461,37910,69.4581,0.50,0.80
4,121,The Lord of the Rings: The Two Towers,0.572523,8.400,22935,17.8899,0.50,0.60
5,120,The Lord of the Rings: The Fellowship of the Ring,0.571523,8.424,26400,23.6040,0.50,0.55
6,603,The Matrix,0.557734,8.232,26818,18.1917,0.50,0.45
7,293660,Deadpool,0.548778,7.623,31933,12.7692,0.50,0.70
8,37724,Skyfall,0.536367,7.254,15618,8.6382,0.50,0.90
9,155,The Dark Knight,0.507233,8.523,34439,67.4752,0.25,0.90



Top picks for The Dark Knight (155)


,id,title,score,vote_average,vote_count,popularity,genre_overlap,year_proximity
0,49026,The Dark Knight Rises,0.834950,7.787,23557,17.1291,1.00,0.80
1,2112,Payback,0.723227,6.827,1900,2.9530,1.00,0.55
2,272,Batman Begins,0.702465,7.717,21808,14.4361,0.75,0.85
3,680,Pulp Fiction,0.672126,8.487,29102,17.3226,0.75,0.30
4,2118,L.A. Confidential,0.663412,7.801,5234,6.1879,0.75,0.45
5,9799,The Fast and the Furious,0.615366,6.996,10407,0.9797,0.75,0.65
6,550,Fight Club,0.583876,8.438,30785,41.2520,0.50,0.55
7,2142,Cop Land,0.580139,6.792,1684,2.5421,0.75,0.45
8,2153,The Driver,0.566395,7.174,477,3.5360,0.75,0.00
9,497,The Green Mile,0.566104,8.503,18387,15.8630,0.50,0.55



Top picks for Breaking Bad (1396)


,id,name,score,vote_average,vote_count,popularity,genre_overlap,year_proximity
0,1405,Dexter,0.702668,8.200,4923,160.9545,0.5,0.90
1,2288,Prison Break,0.654221,8.076,5595,98.8865,0.5,0.85
2,19885,Sherlock,0.605930,8.512,5937,22.7569,0.5,0.90
3,1436,Justified,0.600712,8.000,776,15.3028,0.5,0.90
4,60622,Fargo,0.594821,8.293,2986,35.4579,0.5,0.70
5,60059,Better Call Saul,0.589778,8.694,5870,35.3966,0.5,0.65
6,1438,The Wire,0.588453,8.600,2445,26.3615,0.5,0.70
7,1398,The Sopranos,0.588035,8.600,3121,47.1923,0.5,0.55
8,62560,Mr. Robot,0.578403,8.251,4974,19.1474,0.5,0.65
9,99494,Flower of Evil,0.543905,8.220,561,5.5790,0.5,0.40



Top picks for Game of Thrones (1399)


,id,name,score,vote_average,vote_count,popularity,genre_overlap,year_proximity
0,1402,The Walking Dead,0.775191,8.100,17202,71.7016,0.75,0.95
1,48866,The 100,0.732507,7.900,8370,36.4389,0.75,0.85
2,71912,The Witcher,0.711574,8.010,6267,30.8196,0.75,0.60
3,94997,House of the Dragon,0.698411,8.321,5487,33.4443,0.75,0.45
4,1622,Supernatural,0.651901,8.301,7998,117.0018,0.50,0.70
5,44217,Vikings,0.639464,8.093,7309,70.6627,0.50,0.90
6,66732,Stranger Things,0.620251,8.596,18729,64.6448,0.50,0.75
7,60735,The Flash,0.606167,7.769,11383,56.6385,0.50,0.85
8,1405,Dexter,0.557994,8.219,4922,161.4200,0.25,0.75
9,46331,Under the Dome,0.531492,7.157,3741,12.7605,0.50,0.90


In [12]:
# Simple evaluation harness across seeds
def evaluate_movie_seeds(seed_ids: List[int], k: int = 10) -> pd.DataFrame:
    rows = []
    for sid in seed_ids:
        df = recommend_for_movie(sid, k=k)
        if df.empty:
            continue
        rows.append({
            'seed': sid,
            'avg_vote': float(df['vote_average'].mean()),
            'avg_votes': float(df['vote_count'].mean()),
            'avg_overlap': float(df['genre_overlap'].mean()),
            'avg_year_prox': float(df['year_proximity'].mean()),
            'avg_score': float(df['score'].mean()),
        })
    return pd.DataFrame(rows)

eval_df = evaluate_movie_seeds(list(MOVIE_SEEDS.values()), k=10)
display(eval_df)

,seed,avg_vote,avg_votes,avg_overlap,avg_year_prox,avg_score
0,157336,7.7128,20706.9,0.525,0.720,0.561961
1,27205,7.8658,27416.0,0.550,0.760,0.593750
2,155,7.6522,14334.1,0.750,0.515,0.650806


## Integration notes
- We can move `recommend_for_movie` and `recommend_for_tv` into the Flask microservice (`recommendation_service/app.py`).
- Expose endpoints like `/api/reco/movie/<movie_id>` and `/api/reco/tv/<tv_id>` returning top-N items.
- This keeps TS server as a hedged fallback while Python serves first-class blended results.
- Tune `WEIGHTS` and `QUALITY_FLOORS` here and copy values to the service for consistent behavior.

## Improving recommendation quality: signals and strategy

We’ll add more content signals and a diversity-aware re-ranker to get unique-yet-relevant results:
- Signals: genre overlap, keyword overlap, cast overlap, director/creator overlap, franchise/collection boost, rating, popularity (diminishing), year proximity.
- Floors: minimum vote_count and vote_average to keep quality high.
- Diversity: Maximal Marginal Relevance (MMR) over genres+keywords to avoid near-duplicates and surface variety.
- Optional text similarity: can be enabled later via sentence-transformers for synopsis/title embeddings (not required to run).

In [13]:
# Extra feature extractors: keywords, cast/crew, collections/franchise
def get_keywords(item):
    # movies: in details under 'keywords', TV: sometimes in 'keywords' via other endpoints; fallback to []
    kws = []
    if isinstance(item.get('keywords'), dict) and 'keywords' in item['keywords']:
        kws = [k.get('name','').lower() for k in item['keywords']['keywords'] if isinstance(k, dict)]
    elif isinstance(item.get('keywords'), list):
        kws = [k.get('name','').lower() for k in item['keywords'] if isinstance(k, dict)]
    return [k for k in kws if k]

def get_cast_ids(item):
    credits = item.get('credits') or item.get('aggregate_credits') or {}
    cast = credits.get('cast') or []
    return [int(c.get('id')) for c in cast if c.get('id')]

def get_director_ids(item):
    credits = item.get('credits') or {}
    crew = credits.get('crew') or []
    return [int(m.get('id')) for m in crew if m.get('job') == 'Director' and m.get('id')]

def get_creator_ids(tv_details):
    creators = tv_details.get('created_by') or []
    return [int(c.get('id')) for c in creators if c.get('id')]

def in_same_collection(item_a, item_b):
    ca = (item_a.get('belongs_to_collection') or {}).get('id')
    cb = (item_b.get('belongs_to_collection') or {}).get('id')
    return bool(ca and cb and ca == cb)

In [14]:
# Advanced scoring with additional signals and optional boosts
ADV_WEIGHTS = {
    'genre_overlap': 0.40,
    'keyword_overlap': 0.18,
    'cast_overlap': 0.12,
    'director_creator_overlap': 0.08,
    'year_proximity': 0.08,
    'rating': 0.09,
    'popularity': 0.05,
}

BOOSTS = {
    'same_collection': 0.08,
}

def jaccard_overlap(a, b):
    sa, sb = set(a), set(b)
    if not sa or not sb:
        return 0.0
    inter = len(sa & sb)
    uni = len(sa | sb)
    return inter / uni if uni else 0.0

def score_candidate_advanced(source: Dict[str, Any], cand: Dict[str, Any]) -> Tuple[float, Dict[str, float]]:
    va = float(cand.get('vote_average') or 0.0)
    vc = float(cand.get('vote_count') or 0.0)
    if vc < QUALITY_FLOORS['vote_count'] or va < QUALITY_FLOORS['vote_average']:
        return 0.0, {'floor_filter': 1.0}
    
    # base signals
    src_genres = get_genre_ids(source)
    tgt_genres = get_genre_ids(cand)
    genre_term = clamp(genre_overlap(src_genres, tgt_genres) / 4.0, 0.0, 1.0)
    
    src_year = get_year(source)
    tgt_year = get_year(cand)
    if src_year and tgt_year:
        year_diff = abs(src_year - tgt_year)
        year_term = 1.0 - clamp(year_diff / 20.0, 0.0, 1.0)
    else:
        year_term = 0.5
    
    rating_term = clamp((va - 5.0) / 3.0, 0.0, 1.0)
    pop_term = clamp(float(cand.get('popularity') or 0.0) / 200.0, 0.0, 1.0)
    
    # new signals
    kw_src = get_keywords(source)
    kw_tgt = get_keywords(cand)
    kw_term = clamp(jaccard_overlap(kw_src, kw_tgt), 0.0, 1.0)
    
    cast_src = get_cast_ids(source)
    cast_tgt = get_cast_ids(cand)
    cast_term = clamp(jaccard_overlap(cast_src, cast_tgt), 0.0, 1.0)
    
    # director (movies) or creators (tv)
    if 'title' in source or 'original_title' in source:
        dir_src = get_director_ids(source)
        dir_tgt = get_director_ids(cand)
    else:
        dir_src = get_creator_ids(source) if isinstance(source, dict) else []
        dir_tgt = get_creator_ids(cand) if isinstance(cand, dict) else []
    dir_term = clamp(jaccard_overlap(dir_src, dir_tgt), 0.0, 1.0)
    
    components = {
        'genre_overlap': genre_term,
        'keyword_overlap': kw_term,
        'cast_overlap': cast_term,
        'director_creator_overlap': dir_term,
        'year_proximity': year_term,
        'rating': rating_term,
        'popularity': pop_term,
    }
    score = sum(ADV_WEIGHTS[k] * components[k] for k in ADV_WEIGHTS)
    
    # franchise/collection boost (movies only)
    if in_same_collection(source, cand):
        score += BOOSTS['same_collection']
    
    return float(score), components

In [15]:
# MMR (Maximal Marginal Relevance) for diversity
def mmr_diversify(items: List[Dict[str, Any]],
                   score_key: str = 'score',
                   lambda_weight: float = 0.75,
                   top_k: int = 20) -> List[Dict[str, Any]]:
    """
    items: list of dicts with precomputed 'score' and feature vectors (genres+keywords).
    lambda_weight: balance between relevance and diversity (0..1).
    """
    # Precompute simple set features for similarity
    def feature_set(x):
        return set(get_genre_ids(x)) | set(get_keywords(x))
    
    feats = [feature_set(x) for x in items]
    selected = []
    candidates = list(range(len(items)))
    
    def sim(i, j):
        a, b = feats[i], feats[j]
        if not a or not b:
            return 0.0
        inter = len(a & b)
        uni = len(a | b)
        return inter / uni if uni else 0.0
    
    while candidates and len(selected) < top_k:
        best_idx = None
        best_val = -1e9
        for i in candidates:
            rel = items[i].get(score_key, 0.0)
            if not selected:
                val = rel
            else:
                max_sim = max(sim(i, j) for j in selected)
                val = lambda_weight * rel - (1 - lambda_weight) * max_sim
            if val > best_val:
                best_val = val
                best_idx = i
        selected.append(best_idx)
        candidates.remove(best_idx)
    return [items[i] for i in selected]

In [16]:
# v2 recommenders using advanced scoring + MMR
def recommend_for_movie_v2(movie_id: int, k: int = 20, pool_limit: int = 150) -> pd.DataFrame:
    src = get_movie_details(movie_id)
    sim = get_movie_similar(movie_id)
    rec = get_movie_recommendations(movie_id)
    pool = {m['id']: m for m in (sim + rec) if isinstance(m, dict) and m.get('id')}
    
    rows = []
    for cid, cand in pool.items():
        if cid == movie_id:
            continue
        score, comps = score_candidate_advanced(src, cand)
        if score <= 0:
            continue
        rows.append({
            'id': cid,
            'title': cand.get('title') or cand.get('name'),
            'score': float(score),
            'vote_average': cand.get('vote_average'),
            'vote_count': cand.get('vote_count'),
            'popularity': cand.get('popularity'),
            'genre_overlap': comps['genre_overlap'],
            'year_proximity': comps['year_proximity'],
        })
    
    if not rows:
        return pd.DataFrame(columns=['id','title','score'])
    rows = sorted(rows, key=lambda r: r['score'], reverse=True)[:pool_limit]
    diversified = mmr_diversify(rows, score_key='score', lambda_weight=0.78, top_k=k)
    return pd.DataFrame(diversified).reset_index(drop=True)
    
def recommend_for_tv_v2(tv_id: int, k: int = 20, pool_limit: int = 150) -> pd.DataFrame:
    src = get_tv_details(tv_id)
    sim = get_tv_similar(tv_id)
    rec = get_tv_recommendations(tv_id)
    pool = {m['id']: m for m in (sim + rec) if isinstance(m, dict) and m.get('id')}
    
    rows = []
    for cid, cand in pool.items():
        if cid == tv_id:
            continue
        score, comps = score_candidate_advanced(src, cand)
        if score <= 0:
            continue
        rows.append({
            'id': cid,
            'name': cand.get('name') or cand.get('title'),
            'score': float(score),
            'vote_average': cand.get('vote_average'),
            'vote_count': cand.get('vote_count'),
            'popularity': cand.get('popularity'),
            'genre_overlap': comps['genre_overlap'],
            'year_proximity': comps['year_proximity'],
        })
    
    if not rows:
        return pd.DataFrame(columns=['id','name','score'])
    rows = sorted(rows, key=lambda r: r['score'], reverse=True)[:pool_limit]
    diversified = mmr_diversify(rows, score_key='score', lambda_weight=0.78, top_k=k)
    return pd.DataFrame(diversified).reset_index(drop=True)

In [17]:
# Quick comparison: v1 vs v2 on a few seeds
def compare_v1_v2(movie_seeds: Dict[str,int], tv_seeds: Dict[str,int], k: int = 10):
    for name, mid in movie_seeds.items():
        print(f"\nMovie: {name} ({mid})")
        v1 = recommend_for_movie(mid, k=k)
        v2 = recommend_for_movie_v2(mid, k=k)
        print("v1 head:")
        display(v1.head(5))
        print("v2 head:")
        display(v2.head(5))
    for name, tid in tv_seeds.items():
        print(f"\nTV: {name} ({tid})")
        v1 = recommend_for_tv(tid, k=k)
        v2 = recommend_for_tv_v2(tid, k=k)
        print("v1 head:")
        display(v1.head(5))
        print("v2 head:")
        display(v2.head(5))
        
# run a quick comparison (uses existing MOVIE_SEEDS / TV_SEEDS from earlier cells)
compare_v1_v2(MOVIE_SEEDS, TV_SEEDS, k=10)


Movie: Interstellar (157336)
v1 head:


,id,title,score,vote_average,vote_count,popularity,genre_overlap,year_proximity
0,286217,The Martian,0.712292,7.700,20443,16.1315,0.75,0.95
1,27205,Inception,0.599646,8.370,37994,28.0656,0.50,0.80
2,118340,Guardians of the Galaxy,0.597202,7.905,28893,6.8599,0.50,1.00
3,76341,Mad Max: Fury Road,0.573161,7.626,23382,11.5439,0.50,0.95
4,1124,The Prestige,0.566132,8.204,16817,8.7604,0.50,0.60


v2 head:


,id,title,score,vote_average,vote_count,popularity,genre_overlap,year_proximity
0,286217,The Martian,0.461033,7.700,20443,16.1315,0.75,0.95
1,118340,Guardians of the Galaxy,0.368865,7.905,28893,6.8599,0.50,1.00
2,27205,Inception,0.361016,8.370,37994,28.0656,0.50,0.80
3,76341,Mad Max: Fury Road,0.357666,7.626,23382,11.5439,0.50,0.95
4,49047,Gravity,0.343009,7.162,15813,8.5962,0.50,0.95



Movie: Inception (27205)
v1 head:


...

,id,name,score,vote_average,vote_count,popularity,genre_overlap,year_proximity
0,1405,Dexter,0.402239,8.200,4923,160.9545,0.5,0.90
1,2288,Prison Break,0.382722,8.076,5595,98.8865,0.5,0.85
2,19885,Sherlock,0.367689,8.512,5937,22.7569,0.5,0.90
3,1436,Justified,0.365826,8.000,776,15.3028,0.5,0.90
4,60622,Fargo,0.354864,8.293,2986,35.4579,0.5,0.70



TV: Game of Thrones (1399)
v1 head:


,id,name,score,vote_average,vote_count,popularity,genre_overlap,year_proximity
0,1402,The Walking Dead,0.775191,8.100,17202,71.7016,0.75,0.95
1,48866,The 100,0.732507,7.900,8370,36.4389,0.75,0.85
2,71912,The Witcher,0.711574,8.010,6267,30.8196,0.75,0.60
3,94997,House of the Dragon,0.698411,8.321,5487,33.4443,0.75,0.45
4,1622,Supernatural,0.651901,8.301,7998,117.0018,0.50,0.70


v2 head:


,id,name,score,vote_average,vote_count,popularity,genre_overlap,year_proximity
0,1402,The Walking Dead,0.483925,8.100,17202,71.7016,0.75,0.95
1,48866,The 100,0.464110,7.900,8370,36.4389,0.75,0.85
2,71912,The Witcher,0.445705,8.010,6267,30.8196,0.75,0.60
3,94997,House of the Dragon,0.434361,8.321,5487,33.4443,0.75,0.45
4,44217,Vikings,0.379666,8.093,7309,70.6627,0.50,0.90
